# SAC `training_intensity` → round-robin weights (RLlib new API stack)

Reproduces RLlib's `calculate_rr_weights()` and derives, for a sweep of
`training_intensity` values, how many env steps are sampled and how many replay
transitions are learned on per training iteration.

**Scope: new API stack only** (`enable_rl_module_and_learner=True`,
`enable_env_runner_and_connector_v2=True`) — the stack this repo trains on.

### Where this lives in RLlib (Ray 2.52.1)

`SAC` is a subclass of `DQN` (`ray/rllib/algorithms/sac/sac.py:561`) and does not
override `training_step`, so SAC runs DQN's loop
(`ray/rllib/algorithms/dqn/dqn.py:626`), which dispatches to
`_training_step_new_api_stack()` (`dqn.py:648`):

```python
store_weight, sample_and_train_weight = calculate_rr_weights(self.config)   # dqn.py:650

for _ in range(store_weight):                       # dqn.py:653
    episodes, env_runner_results = synchronous_parallel_sample(...)         # dqn.py:656
    self.local_replay_buffer.add(episodes)                                  # dqn.py:670

if current_ts >= self.config.num_steps_sampled_before_learning_starts:      # dqn.py:681
    for _ in range(sample_and_train_weight):                                # dqn.py:683
        episodes = self.local_replay_buffer.sample(
            num_items=self.config.total_train_batch_size, ...)              # dqn.py:687
        learner_results = self.learner_group.update(episodes=episodes, ...) # dqn.py:715
```

So per iteration you run `store_weight` sampling rounds (each appended to the
`EpisodeReplayBuffer`) and `sample_and_train_weight` gradient updates, each on
`total_train_batch_size` replayed transitions.

## 1. Config parameters

Only these four config fields feed the rr-weight formula
(`dqn.py:568-592`): train batch size, rollout fragment length, envs per runner,
and number of env runners.

In this repo they come from `adv_building_gym/ray/training/select_model.py`:
`train_batch_size_per_learner = sac_replay_batch_size` (line 82),
`rollout_fragment_length = sac_rollout_length or env_episode_length` (line 100),
and `training_intensity = sac_training_intensity` (line 90); `num_env_runners`
is derived from the SLURM CPU count in
`adv_building_gym/ray/training/common_model_config.py`.

In [19]:
import numpy as np
import pandas as pd

# --- RLlib config fields that enter the formula ---------------------------
# Trailing comments give the SACConfig() default (verified on Ray 2.52.1).
TOTAL_TRAIN_BATCH_SIZE = 32    # config.total_train_batch_size — SAC default: 256
#   (new API stack: train_batch_size_per_learner * (num_learners or 1), see
#    algorithm_config.py:4174; SAC defaults 256 * (0 or 1) -> 256)
ROLLOUT_FRAGMENT_LENGTH = 144   # config.rollout_fragment_length — SAC default: "auto"
N_STEP = 3                      # config.n_step — SAC default: 1; used only when rollout length == "auto"
NUM_ENVS_PER_ENV_RUNNER = 1     # config.num_envs_per_env_runner — SAC default: 1
NUM_ENV_RUNNERS = 4             # config.num_env_runners (remote runners) — SAC default: 0

# --- Sweep -----------------------------------------------------------------
# config.training_intensity — SAC default: None (-> weights [1, 1], see section 5).
TRAINING_INTENSITIES = [1, 2, 4, 8, 10, 12, 16, 20, 32, 64, 128, 256]
HIGHLIGHT_INTENSITY = 10        # value to mark in the table (e.g. the configured one)

## 2. Ports of the RLlib formulas

Faithful ports of the three pieces of Ray 2.52.1 that decide the weights.

* `get_rollout_fragment_length()` — `SACConfig` overrides the base
  implementation (`sac/sac.py:500-508`): with `"auto"` it returns `n_step`
  (not the base class' `total_train_batch_size / (num_envs * num_runners)`);
  otherwise the configured int, verbatim.
* `sampler_footprint()` — the denominator of the native ratio
  (`dqn.py:578-584`). It multiplies by `max(num_env_runners + 1, 1)`, i.e. it
  *assumes* the local env runner samples too.
* `calculate_rr_weights()` — `dqn.py:568-592`.

In [20]:
def get_rollout_fragment_length(rollout_fragment_length, n_step):
    """Port of SACConfig.get_rollout_fragment_length() (ray/rllib/algorithms/sac/sac.py:500)."""
    if rollout_fragment_length == "auto":
        return n_step[1] if isinstance(n_step, (tuple, list)) else n_step
    return rollout_fragment_length


def sampler_footprint():
    """Env-step count RLlib *assumes* per sampling round (dqn.py:578-584).

    The `+1` mirrors RLlib: it counts the local env runner as a sampler and
    avoids division by zero. See section 4 for why that is optimistic here.
    """
    return (
        get_rollout_fragment_length(ROLLOUT_FRAGMENT_LENGTH, N_STEP)
        * NUM_ENVS_PER_ENV_RUNNER
        * max(NUM_ENV_RUNNERS + 1, 1)
    )


def native_ratio():
    """train_batch / assumed-env-steps-per-rollout. Constant across intensities."""
    return TOTAL_TRAIN_BATCH_SIZE / sampler_footprint()


def calculate_rr_weights(training_intensity):
    """Port of dqn.py::calculate_rr_weights -> [store_weight, sample_and_train_weight]."""
    if not training_intensity:
        return [1, 1]

    sample_and_train_weight = training_intensity / native_ratio()

    # NOTE: np.round is banker's rounding (round-half-to-even), so 22.5 -> 22.
    if sample_and_train_weight < 1:
        return [int(np.round(1 / sample_and_train_weight)), 1]
    return [1, int(np.round(sample_and_train_weight))]

## 3. Actual sampling footprint

What the loop *really* collects differs from `sampler_footprint()`. With at
least one remote env runner, `synchronous_parallel_sample` dispatches with
`local_env_runner=False` (`ray/rllib/execution/rollout_ops.py:105-115`), so only
the **remote** runners sample; the local one is used only when there are no
remote workers (`rollout_ops.py:101-103`). Each remote `sample()` call collects
`get_rollout_fragment_length(worker_index) * num_envs` timesteps
(`ray/rllib/env/single_agent_env_runner.py:217-226`, `batch_mode="truncate_episodes"`).

So the realized replay-to-sample ratio is `training_intensity * (N + 1) / N` for
`N` remote runners — RLlib divides by `N + 1` but only `N` runners sample.

In [21]:
def actual_sampled_steps():
    """Env steps really collected per sampling round (remote runners only)."""
    rollout_len = get_rollout_fragment_length(ROLLOUT_FRAGMENT_LENGTH, N_STEP)
    sampling_runners = NUM_ENV_RUNNERS if NUM_ENV_RUNNERS > 0 else 1
    return rollout_len * NUM_ENVS_PER_ENV_RUNNER * sampling_runners


rollout_len = get_rollout_fragment_length(ROLLOUT_FRAGMENT_LENGTH, N_STEP)
assumed_steps = sampler_footprint()
sampled_steps = actual_sampled_steps()

# dtype=object keeps the ints from being widened to float by the ratio entry.
summary = pd.Series(
    {
        "total_train_batch_size": TOTAL_TRAIN_BATCH_SIZE,
        "rollout_fragment_length (config)": ROLLOUT_FRAGMENT_LENGTH,
        "rollout_fragment_length (resolved)": rollout_len,
        "n_step": N_STEP,
        "num_envs_per_env_runner": NUM_ENVS_PER_ENV_RUNNER,
        "num_env_runners": NUM_ENV_RUNNERS,
        "assumed env steps / round (formula, x(N+1))": assumed_steps,
        "actual env steps / round (remote only, xN)": sampled_steps,
        "native_ratio = batch / assumed steps": f"{native_ratio():.4f}",
    },
    name="value",
    dtype=object,
)
summary.to_frame().style.set_table_styles(
    [{"selector": "td", "props": [("text-align", "right"), ("padding", "2px 10px")]}]
)

,value
total_train_batch_size,32
rollout_fragment_length (config),144
rollout_fragment_length (resolved),144
n_step,3
num_envs_per_env_runner,1
num_env_runners,4
"assumed env steps / round (formula, x(N+1))",720
"actual env steps / round (remote only, xN)",576
native_ratio = batch / assumed steps,0.0444


## 4. Sweep table

Per `training_intensity`:

| column | meaning |
| --- | --- |
| `store_w` / `s&t_w` | the two round-robin weights returned by `calculate_rr_weights` |
| `s&t_w exact` | the pre-rounding value `training_intensity / native_ratio` |
| `env steps/iter` | `store_w x` actual sampled steps (remote runners only) |
| `replayed/iter` | `s&t_w x total_train_batch_size` |
| `target r:s` | requested replay:sample ratio, i.e. `training_intensity` |
| `realized r:s` | `replayed/iter ÷ env steps/iter` |
| `updates/iter` | gradient updates per iteration (`= s&t_w`) |
| `UTD` | updates per env step (`updates/iter ÷ env steps/iter`) |

In [22]:
rows = []
for intensity in TRAINING_INTENSITIES:
    store_w, sample_train_w = calculate_rr_weights(intensity)

    env_steps = store_w * sampled_steps
    replayed = sample_train_w * TOTAL_TRAIN_BATCH_SIZE

    rows.append(
        {
            "training_intensity": intensity,
            "store_w": store_w,
            "s&t_w": sample_train_w,
            "s&t_w exact": intensity / native_ratio(),
            "sampled env steps/iter": env_steps,
            "replayed steps/iter": replayed,
            "target r:s": float(intensity),
            "realized r:s": replayed / env_steps,
            "updates/iter": sample_train_w,
            "UTD": sample_train_w / env_steps,
        }
    )

sweep = pd.DataFrame(rows)
sweep

,training_intensity,store_w,s&t_w,s&t_w exact,sampled env steps/iter,replayed steps/iter,target r:s,realized r:s,updates/iter,UTD
0,1,1,22,22.5,576,704,1.0,1.222222,22,0.038194
1,2,1,45,45.0,576,1440,2.0,2.500000,45,0.078125
2,4,1,90,90.0,576,2880,4.0,5.000000,90,0.156250
3,8,1,180,180.0,576,5760,8.0,10.000000,180,0.312500
4,10,1,225,225.0,576,7200,10.0,12.500000,225,0.390625
5,12,1,270,270.0,576,8640,12.0,15.000000,270,0.468750
6,16,1,360,360.0,576,11520,16.0,20.000000,360,0.625000
7,20,1,450,450.0,576,14400,20.0,25.000000,450,0.781250
8,32,1,720,720.0,576,23040,32.0,40.000000,720,1.250000
9,64,1,1440,1440.0,576,46080,64.0,80.000000,1440,2.500000


### Formatted view

Rounding to whole weights is where the requested and realized ratios drift
apart; the highlighted row is `HIGHLIGHT_INTENSITY`.

In [23]:
def highlight_configured(row):
    is_match = row["training_intensity"] == HIGHLIGHT_INTENSITY
    return ["font-weight: bold; background-color: rgba(255, 214, 102, 0.35)" if is_match else ""] * len(row)


styled = (
    sweep.style.hide(axis="index")
    .format(
        {
            "s&t_w exact": "{:.2f}",
            "env steps/iter": "{:,d}",
            "replayed/iter": "{:,d}",
            "target r:s": "{:.1f}",
            "realized r:s": "{:.2f}",
            "UTD": "{:.4f}",
        }
    )
    .apply(highlight_configured, axis=1)
    .set_caption(
        f"rr-weights sweep — batch={TOTAL_TRAIN_BATCH_SIZE}, "
        f"rollout={rollout_len}, envs/runner={NUM_ENVS_PER_ENV_RUNNER}, "
        f"runners={NUM_ENV_RUNNERS}, native_ratio={native_ratio():.4f}"
    )
    .set_table_styles(
        [
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"), ("padding-bottom", "0.5em")]},
            {"selector": "th", "props": [("text-align", "right"), ("padding", "2px 10px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "2px 10px")]},
        ]
    )
)
styled

training_intensity,store_w,s&t_w,s&t_w exact,sampled env steps/iter,replayed steps/iter,target r:s,realized r:s,updates/iter,UTD
1,1,22,22.50,576,704,1.0,1.22,22,0.0382
2,1,45,45.00,576,1440,2.0,2.50,45,0.0781
4,1,90,90.00,576,2880,4.0,5.00,90,0.1562
8,1,180,180.00,576,5760,8.0,10.00,180,0.3125
10,1,225,225.00,576,7200,10.0,12.50,225,0.3906
12,1,270,270.00,576,8640,12.0,15.00,270,0.4688
16,1,360,360.00,576,11520,16.0,20.00,360,0.6250
20,1,450,450.00,576,14400,20.0,25.00,450,0.7812
32,1,720,720.00,576,23040,32.0,40.00,720,1.2500
64,1,1440,1440.00,576,46080,64.0,80.00,1440,2.5000


### Plain-text view

Same table without the HTML styler, for terminals and log files.

In [24]:
plain = sweep.copy()
for column, spec in {
    "s&t_w exact": "{:.2f}",
    "target r:s": "{:.1f}",
    "realized r:s": "{:.2f}",
    "UTD": "{:.4f}",
}.items():
    plain[column] = plain[column].map(spec.format)

print(plain.to_string(index=False))

 training_intensity  store_w  s&t_w s&t_w exact  sampled env steps/iter  replayed steps/iter target r:s realized r:s  updates/iter     UTD
                  1        1     22       22.50                     576                  704        1.0         1.22            22  0.0382
                  2        1     45       45.00                     576                 1440        2.0         2.50            45  0.0781
                  4        1     90       90.00                     576                 2880        4.0         5.00            90  0.1562
                  8        1    180      180.00                     576                 5760        8.0        10.00           180  0.3125
                 10        1    225      225.00                     576                 7200       10.0        12.50           225  0.3906
                 12        1    270      270.00                     576                 8640       12.0        15.00           270  0.4688
                 16        

## 5. Reading the table

* **`training_intensity` is a replay:sample ratio, not an update count.**
  `total_train_batch_size` is the transitions per update, so the gradient
  updates per iteration are `training_intensity / native_ratio`, rounded.
* **UTD ≈ `training_intensity / total_train_batch_size`.** With the formula's
  assumed footprint the batch size cancels exactly; the realized value is
  `(N+1)/N` higher (section 3). Vanilla SAC uses UTD ≈ 1.0, so matching it
  needs `training_intensity ≈ total_train_batch_size`.
* **Rounding is coarse at low intensities.** `np.round` is banker's rounding
  (half-to-even), and once `s&t_w exact < 1` the weights flip to
  `[round(1 / exact), 1]` — sampling several rounds per single update.
* **`training_intensity = None` (RLlib default) gives `[1, 1]`**
  (`dqn.py:570-571`): one sampling round and one update per iteration, i.e. a
  UTD of `1 / env steps per iteration`.